# Pipeline Sinh Dataset Can Thiep Van Toc (Kubric GPU) tren Kaggle -> Hugging Face

Notebook toi uu hoa rieng cho moi truong **Kaggle** (GPU T4 x2 / P100 / L4), **khong can Docker**:

1. Cai dat cac package ho tro dieu phoi (`huggingface_hub`, `pyyaml`, `pandas`, `uv`).
2. Clone repository `p1neapplechoco/kubric`.
3. Thiet lap moi truong **Python 3.11 Native** va **Blender 4.2 LTS (`bpy==4.2.0`)** kem kernel Cycles CUDA bang `uv`.
4. Kiem tra va kich hoat GPU Cycles ben trong Blender.
5. Chay kiem thu vat ly nhanh (Smoke Test 1 scene, khong render).
6. Sinh dataset day du voi cau hinh can thiep:
   - **Can thiep van toc**: 1 huong duy nhat (cap van toc dau o buoc t=0 cho subject, khong ngoai luc sau do).
   - **Vat lieu**: Sampling tu 6 ho vat lieu ghep cap he so ma sat va dan hoi.
   - **3 nhanh mo phong**: `factual`, `counterfactual` (ap van toc dau moi), `subject_removed` (bo subject).
   - **4 loai video cho tung nhanh**: Video (RGB `video.mp4`), Mask (segmentation `mask.mp4`), Depth map (`depth.mp4`), Optical flow (`flow.mp4`).
   - **Chay song song nhieu video cung luc**: Tham so `--workers` toi uu hoa GPU & CPU.
7. Kiem tra ket qua & hien thi preview video truc quan.
8. Upload toan bo dataset len Hugging Face Hub bang Access Token.

## 1. Cai dat Package & Thiet lap Import

In [ ]:
# Cai dat cac package ho tro dieu phoi
!pip install --quiet "huggingface_hub>=0.24" "pyyaml>=6" "pandas>=2" uv

import os
import sys
import json
import time
import shutil
import subprocess
from pathlib import Path
import pandas as pd
import yaml
from huggingface_hub import HfApi

print("Python version (Kaggle host):", sys.version.split()[0])
print("Cac thu vien dieu phoi da san sang.")

## 2. Cau hinh Tham so Sinh Dataset & Hugging Face

In [ ]:
# Thong tin repository
REPO_URL = "https://github.com/p1neapplechoco/kubric"
REPO_REF = "artifacts/data-generation-notebook"  # Nhanh pipeline can thiep van toc
# Tren Kaggle, thu muc ghi mac dinh la /kaggle/working
WORKDIR = "/kaggle/working/kubric-work" if Path("/kaggle/working").exists() else "./kubric-work"

# Tham so sinh dataset
MASTER_SEED = 0
SCENE_COUNT = 4              # So luong scene can sinh (moi scene gom 3 nhanh = 12 videos)
# Khuyen nghi WORKERS theo VRAM: 2-4 (4-8GB), 4-8 (16GB T4/P100), 8-16 (24-48GB L4/A10G), 16-32+ (A100/H100)
WORKERS = 4                  # So luong video / tasks render song song cung luc (ho tro > 4)
RESOLUTION = 512             # Do phan giai video (256, 512, 768, 1024)
SAMPLES_PER_PIXEL = 64       # Cycles spp
REQUIRE_GPU = True           # Bat buoc dung GPU cho Cycles (bao loi neu khong co GPU)

# Thong tin upload len Hugging Face
HF_REPO_ID = "pineapplechoco/good-morning"  # Dinh dang: username/dataset-name
HF_TOKEN = ""                                # Access token co quyen WRITE tren Hugging Face
HF_PRIVATE = True                            # Dat dataset o che do rieng tu (private)

## 3. Clone Repository & Chuan bi Thu muc Lam viec

In [ ]:
work_path = Path(WORKDIR).resolve()
work_path.mkdir(parents=True, exist_ok=True)
repo_dir = work_path / "kubric"

if not (repo_dir / ".git").exists():
    print(f"Dang clone {REPO_URL} (branch: {REPO_REF})...")
    res = subprocess.run(f"git clone --depth 1 --branch {REPO_REF} {REPO_URL} {repo_dir}", shell=True)
    if res.returncode != 0:
        print("Clone branch that bai, tien hanh clone toan bo repo va checkout...")
        subprocess.run(f"git clone {REPO_URL} {repo_dir}", shell=True, check=True)
        subprocess.run(f"cd {repo_dir} && git checkout {REPO_REF}", shell=True, check=False)
else:
    print("Repository da ton tai. Dang cap nhat branch...")
    subprocess.run(f"cd {repo_dir} && git fetch --all && git checkout {REPO_REF} && git pull --ff-only || true", shell=True)

print("Repository da san sang tai:", repo_dir)

## 4. Setup Python 3.11 Native & Blender 4.2 LTS (Khong dung Docker)

Kaggle khong ho tro Docker daemon. Chung ta su dung **`uv`** de:
1. Cai dat cac thu vien do hoa he thong can thiet cho Blender (`libgl1`, `libx11`, `ffmpeg`...).
2. Tao virtualenv doc lap **Python 3.11** (khong anh huong den Python mac dinh cua Kaggle).
3. Cai dat dung phien ban **`bpy==4.2.0`** (Blender 4.2 LTS Cycles ho tro CUDA) va cac dependencies pinned tu `requirements_render.txt`.

In [ ]:
print("[1/3] Cai dat cac goi do hoa he thong...\n")
subprocess.run(
    "sudo apt-get update -y && sudo apt-get install -y --no-install-recommends "
    "libx11-6 libxi6 libxxf86vm1 libxfixes3 libxrender1 libgl1 libglu1-mesa "
    "libsm6 libice6 libxkbcommon0 libegl1 libgomp1 ffmpeg",
    shell=True, check=False
)

print("\n[2/3] Tao moi truong Python 3.11 bang uv...")
venv_dir = repo_dir / ".venv-render"
if not (venv_dir / "bin" / "python").exists():
    subprocess.run(f"uv venv {venv_dir} --python 3.11", shell=True, check=True)

print("\n[3/3] Cai dat bpy==4.2.0 va cac dependencies render vao Python 3.11...")
subprocess.run(
    f"uv pip install --python {venv_dir / 'bin' / 'python'} -r requirements_render.txt",
    shell=True, cwd=str(repo_dir), check=True
)

runner = str(venv_dir / "bin" / "python")
print(f"\nTrinh chay Python 3.11 Native da san sang: {runner}")

# Kiem tra Cycles GPU ben trong Blender
probe_script = (
    "import bpy, json; "
    "p = bpy.context.preferences.addons['cycles'].preferences; "
    "devices = {b: [d.name for d in p.get_devices_for_type(b)] for b in ('OPTIX', 'CUDA')}; "
    "print('BLENDER_PROBE ' + json.dumps({'bpy_version': bpy.app.version_string, 'devices': devices}))"
)
probe_res = subprocess.run([runner, "-c", probe_script], cwd=str(repo_dir), capture_output=True, text=True)
for line in probe_res.stdout.splitlines():
    if line.startswith("BLENDER_PROBE "):
        print("Ket qua kiem tra Blender Cycles:", line[len("BLENDER_PROBE "):])

## 5. Chay Kiem thu Vat ly (Smoke Test, 1 Scene, Khong Render)

In [ ]:
smoke_out = work_path / "smoke_test"
smoke_cmd = f"{runner} scripts/build_velocity_dataset.py --output {smoke_out} --seed 12345 --count 1 --no-render"
env_vars = dict(os.environ, PYTHONPATH=str(repo_dir), TF_CPP_MIN_LOG_LEVEL="3")
subprocess.run(smoke_cmd, shell=True, cwd=str(repo_dir), env=env_vars, check=True)

qc_path = smoke_out / "instances" / "000000" / "qc.json"
gt_path = smoke_out / "instances" / "000000" / "ground_truth.json"
qc = json.loads(qc_path.read_text())
gt = json.loads(gt_path.read_text())

print("Kiem thu vat ly thanh cong!")
print("- QC passed:", qc["report"]["passed"])
print("- So vat the bi va cham:", qc["report"]["metrics"]["factual_struck"])
print("- So vat the bystander khong va cham:", qc["report"]["metrics"]["factual_untouched"])
print("- Hard affected objects:", gt["hard_affected"])

## 6. Sinh Dataset Day Du (Vat ly + Blender GPU Render Song Song)

Lenh duoi day se sinh day du cac scene. Voi `--workers {WORKERS}`, pipeline se:
1. Chay mo phong vat ly song song cac scene tren CPU.
2. Render song song `{WORKERS}` video cung luc tren GPU (moi video chay 1 process rieng biet, an toan bo nho).
3. Xuat day du: **RGB Video** (`video.mp4`), **Mask** (`mask.mp4`), **Depth Map** (`depth.mp4`), **Optical Flow** (`flow.mp4`), du lieu quang hoc `forward_flow.npz`, `depth.npz`, `segmentation.npz` va `tracking.npz`.

In [ ]:
dataset_dir = work_path / "dataset"
gpu_opt = "--require-gpu --strict" if REQUIRE_GPU else ""
build_cmd = (
    f"{runner} scripts/build_velocity_dataset.py --output {dataset_dir} "
    f"--seed {MASTER_SEED} --count {SCENE_COUNT} --workers {WORKERS} --resolution {RESOLUTION} "
    f"--samples {SAMPLES_PER_PIXEL} --layers rgba segmentation depth forward_flow {gpu_opt}"
)

total_videos = SCENE_COUNT * 3
print(f"Bat dau sinh {SCENE_COUNT} scenes ({total_videos} videos) song song voi {WORKERS} workers...")
t0 = time.time()
build_env = dict(os.environ, PYTHONPATH=str(repo_dir), TF_CPP_MIN_LOG_LEVEL="3", KUBRIC_USE_GPU="true", CUDA_MODULE_LOADING="LAZY")
subprocess.run(build_cmd, shell=True, cwd=str(repo_dir), env=build_env, check=True)
print(f"Hoan thanh sinh dataset sau {time.time() - t0:.1f} giay! Thu muc luu tru: {dataset_dir}")

## 7. Kiem tra Output (Video RGB, Mask, Depth Map, Optical Flow, Tracking)

In [ ]:
from IPython.display import HTML, display
import base64

manifest_file = dataset_dir / "manifest.jsonl"
if manifest_file.exists():
    rows = [json.loads(l) for l in manifest_file.read_text().splitlines() if l.strip()]
    df = pd.DataFrame([{
        "index": r["index"],
        "split": r["split"],
        "objects": r["object_count"],
        "subject": r["subject_shape"],
        "struck": ",".join(r["factual_struck"]),
        "untouched": ",".join(r["factual_untouched"]),
        "hard_affected": ",".join(r["hard_affected"]),
        "renders": ",".join(r["rendered_branches"])
    } for r in rows])
    print("Danh sach cac instances da tao:")
    display(df)

    # Kiem tra cac file cua scene dau tien
    first_dir = dataset_dir / rows[0]["path"]
    print(f"\nDanh sach file trong scene 0 ({first_dir}):")
    for p in sorted(first_dir.glob("**/*")):
        if p.is_file():
            print(f"  {p.relative_to(first_dir)} ({p.stat().st_size:,} bytes)")

    # Kiem tra cac video dau ra cho tung branch
    print("\nKiem tra cac video output cho tung nhanh:")
    for branch in ("factual", "counterfactual", "subject_removed"):
        b_dir = first_dir / branch
        if b_dir.exists():
            vid_files = sorted([f.name for f in b_dir.glob("*.mp4")])
            npz_files = sorted([f.name for f in b_dir.glob("*.npz")])
            print(f"  [{branch}] Videos: {vid_files} | Arrays: {npz_files}")

    # Preview 4 video cua nhanh factual (RGB, Mask, Depth, Flow)
    factual_dir = first_dir / "factual"
    if factual_dir.exists():
        html_parts = ["<h3>Preview Video Scene 0 (Nhanh Factual)</h3><div style=\"display: flex; gap: 15px; flex-wrap: wrap;\">"]
        titles = {"video.mp4": "RGB Video", "mask.mp4": "Segmentation Mask", "depth.mp4": "Depth Map", "flow.mp4": "Optical Flow"}
        for fname, title in titles.items():
            vpath = factual_dir / fname
            if vpath.exists():
                b64 = base64.b64encode(vpath.read_bytes()).decode()
                html_parts.append(
                    f"<div style=\"text-align: center;\"><b>{title}</b><br/>"
                    f"<video width=\"240\" height=\"240\" controls autoplay loop muted>"
                    f"<source src=\"data:video/mp4;base64,{b64}\" type=\"video/mp4\">"
                    f"</video></div>"
                )
        html_parts.append("</div>")
        display(HTML("".join(html_parts)))


## 8. Upload Dataset len Hugging Face Hub

In [ ]:
if not HF_REPO_ID or "/" not in HF_REPO_ID:
    print("Vui long nhap HF_REPO_ID tai muc 2 theo dinh dang: username/dataset-name")
elif not HF_TOKEN:
    print("Vui long nhap HF_TOKEN (write access token) tai muc 2 de tien hanh upload")
else:
    publish_args = [
        sys.executable, "scripts/publish_velocity_dataset.py",
        "--output", str(dataset_dir),
        "--repo-id", HF_REPO_ID,
        "--seed", str(MASTER_SEED),
    ] + ([] if HF_PRIVATE else ["--public"])

    print(f"Dang upload toan bo dataset len Hugging Face: {HF_REPO_ID}...")
    pub_env = dict(os.environ, HF_TOKEN=HF_TOKEN)
    subprocess.run(publish_args, cwd=str(repo_dir), env=pub_env, check=True)
    print(f"Upload hoan tat! Truy cap dataset tai: https://huggingface.co/datasets/{HF_REPO_ID}")